In [128]:
import numpy as np
import pandas as pd

In [129]:
from sklearn.model_selection import KFold,cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.decomposition import PCA

In [130]:
df= pd.read_csv('C:/Users/8429s/OneDrive/Desktop/Placement/Projects/Real Estate Price Prediction and Recommendation System/Data_Clean/flats_and_house_feature_selected_v2.csv')

In [131]:
df.head(5)

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 4,0.58,2,2,1,Moderately Old,67.0,0,0,1,Low,Low Floor
1,flat,sector 77,1.25,3,4,3,Relatively New,1310.0,1,0,1,Low,Mid Floor
2,flat,sector 71,1.00,3,3,3+,Relatively New,1344.0,0,0,2,Low,Mid Floor
3,flat,sector 37c,1.20,3,3,3,Under Construction,1647.0,0,0,1,Low,Mid Floor
4,flat,sector 81,2.20,3,3,3+,Relatively New,1570.0,1,0,0,Low,Mid Floor


In [132]:
df['furnishing_type'].value_counts()

furnishing_type
1    2330
2     971
0     183
Name: count, dtype: int64

In [133]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished
df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [134]:
df.head(5)

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 4,0.58,2,2,1,Moderately Old,67.0,0,0,semifurnished,Low,Low Floor
1,flat,sector 77,1.25,3,4,3,Relatively New,1310.0,1,0,semifurnished,Low,Mid Floor
2,flat,sector 71,1.00,3,3,3+,Relatively New,1344.0,0,0,furnished,Low,Mid Floor
3,flat,sector 37c,1.20,3,3,3,Under Construction,1647.0,0,0,semifurnished,Low,Mid Floor
4,flat,sector 81,2.20,3,3,3+,Relatively New,1570.0,1,0,unfurnished,Low,Mid Floor


In [135]:
X = df.drop(columns=['price'])
y = df['price']

In [136]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

### Applying Ordinal Encoding to categorical column and StandardScaler to numerical columns

In [141]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [142]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num',StandardScaler(),['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat',OrdinalEncoder( handle_unknown='use_encoded_value',
    unknown_value=-1),columns_to_encode)
],remainder='passthrough'
)

In [143]:
#Creating the pipline
pipline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())
])

In [144]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipline, X, y_transformed, cv=kfold, scoring='r2',error_score='raise')

In [146]:
scores.mean(),scores.std()

(np.float64(0.7343437295812951), np.float64(0.03865792221752455))

In [147]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [148]:
pipline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [149]:
y_pred = pipline.predict(X_test)
y_pred = np.expm1(y_pred)

In [150]:
mean_absolute_error(np.expm1(y_test),y_pred)

1.0129219211896736

In [151]:
def scorer(model_name,model):

    output=[]
    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor',preprocessor),
        ('regressor',model)
    ]) 
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output 


In [117]:
!pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [152]:
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

In [153]:
model_dict={
    'linear_regression':LinearRegression(),
    'svr' : SVR(),
    'Ridge': Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [154]:
model_output = []
for model_name , model in model_dict.items():
    model_output.append(scorer(model_name,model))

In [ ]:
model_output

[['linear_regression', np.float64(0.8595095009461049), 0.72659856636032],
 ['svr', np.float64(0.8882987928104502), 0.5905227644912572],
 ['Ridge', np.float64(0.8598544348152359), 0.7313241222797008],
 ['LASSO', np.float64(-0.006061365455360934), 1.6051238249053204],
 ['decision tree', np.float64(0.7802286676054638), 0.7215810239602984],
 ['random forest', np.float64(0.8747727309864695), 0.580380349008559],
 ['extra trees', np.float64(0.8872153040731006), 0.5723310092909397],
 ['gradient boosting', np.float64(0.8609244007094464), 0.6470259070647427],
 ['adaboost', np.float64(0.735314436468326), 0.9464296710639051],
 ['mlp', np.float64(0.8864329628726919), 0.5906404150404797],
 ['xgboost', np.float64(0.8952327636273567), 0.5705533347837894]]

In [155]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.895253,0.513178
5,random forest,0.885993,0.575553
6,extra trees,0.872136,0.616581
7,gradient boosting,0.877433,0.618831
4,decision tree,0.788509,0.705747
9,mlp,0.802804,0.802260
8,adaboost,0.755441,0.874200
1,svr,0.760670,0.942584
2,Ridge,0.734348,1.012845
0,linear_regression,0.734344,1.012922


### OneHotEncoding

In [156]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder( handle_unknown='use_encoded_value',
    unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['sector','agePossession','furnishing_type'])
    ], 
    remainder='passthrough'
)

In [157]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [158]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [159]:
scores.mean()

np.float64(0.8583690782433038)

In [160]:
scores.std()

np.float64(0.02473960021381986)

In [161]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [162]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [163]:
y_pred = pipeline.predict(X_test)

e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [164]:
y_pred = np.expm1(y_pred)

In [165]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.7329119676550991

In [166]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [167]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [168]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
e:\Mtech Projects\Virtual En

In [169]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [172]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.901088,0.532381
5,random forest,0.895351,0.548562
6,extra trees,0.896551,0.552548
7,gradient boosting,0.879303,0.613803
9,mlp,0.872384,0.641055
4,decision tree,0.811992,0.732848
0,linear_reg,0.858369,0.732912
2,ridge,0.858615,0.740128
8,adaboost,0.760116,0.889330
1,svr,0.765215,0.933789


### OneHotEncoding with PCA

In [176]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['sector','agePossession'])
    ], 
    remainder='passthrough'
)

In [177]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [178]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [179]:
scores.mean()

np.float64(0.054099469652641875)

In [180]:
scores.std()

np.float64(0.02771354373125697)

In [181]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [182]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [183]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
e:\Mtech Projects\Virtual Environment\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
e:\Mtech Projects\Virtual En

In [184]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.757120,0.766304
6,extra trees,0.733392,0.793326
4,decision tree,0.684005,0.850185
10,xgboost,0.596898,0.933875
7,gradient boosting,0.612862,1.050105
1,svr,0.226948,1.434292
9,mlp,0.222474,1.479889
8,adaboost,0.281372,1.527604
3,LASSO,0.052078,1.573772
2,ridge,0.054099,1.576571


### Targer Encoder

In [186]:
!pip install category_encoders


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [189]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [190]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [191]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [192]:
scores.mean(),scores.std()

(np.float64(0.8287288072425854), np.float64(0.026989883797781878))

In [193]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [194]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [195]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [196]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.905458,0.506447
5,random forest,0.904771,0.517707
6,extra trees,0.903932,0.518470
7,gradient boosting,0.892938,0.575953
4,decision tree,0.821234,0.680952
9,mlp,0.848909,0.685200
8,adaboost,0.825752,0.782685
0,linear_reg,0.828729,0.801180
2,ridge,0.828750,0.801532
1,svr,0.777328,0.916724


### Hyperparameter Tuning

In [197]:
from sklearn.model_selection import GridSearchCV

In [198]:
from xgboost import XGBRegressor

param_grid = {
    'regressor__n_estimators': [100, 200, 300, 500],
    'regressor__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'regressor__max_depth': [3, 5, 7, 10],
    'regressor__min_child_weight': [1, 3, 5],
    'regressor__subsample': [0.6, 0.8, 1.0],
    'regressor__colsample_bytree': [0.6, 0.8, 1.0],
    'regressor__gamma': [0, 0.1, 0.3, 0.5],
    'regressor__reg_alpha': [0, 0.1, 1],
    'regressor__reg_lambda': [1, 3, 5]
}

In [216]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
    ], 
    remainder='passthrough'
)

In [217]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor())
])

In [218]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

In [219]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=100,
    cv=kfold,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=2,
    error_score='raise'
)

search.fit(X, y_transformed)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


,estimator,"Pipeline(step...=None, ...))])"
,param_distributions,"{'regressor__colsample_bytree': [0.6, 0.8, ...], 'regressor__gamma': [0, 0.1, ...], 'regressor__learning_rate': [0.01, 0.05, ...], 'regressor__max_depth': [3, 5, ...], ...}"
,n_iter,100
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,'raise'


In [220]:
final_pipe = search.best_estimator_

In [221]:
search.best_params_

{'regressor__subsample': 0.6,
 'regressor__reg_lambda': 3,
 'regressor__reg_alpha': 0,
 'regressor__n_estimators': 300,
 'regressor__min_child_weight': 3,
 'regressor__max_depth': 10,
 'regressor__learning_rate': 0.05,
 'regressor__gamma': 0,
 'regressor__colsample_bytree': 0.6}

In [222]:
search.best_score_

np.float64(0.9034979994885759)

In [223]:
final_pipe.fit(X,y_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [226]:
import pickle

with open('C:/Users/8429s/OneDrive/Desktop/Placement/Projects/Real Estate Price Prediction and Recommendation System/model/pipeline.pkl', 'wb') as file:
    pickle.dump(final_pipe, file)

In [228]:
X

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 4,2,2,1,Moderately Old,67.0,0,0,semifurnished,Low,Low Floor
1,flat,sector 77,3,4,3,Relatively New,1310.0,1,0,semifurnished,Low,Mid Floor
2,flat,sector 71,3,3,3+,Relatively New,1344.0,0,0,furnished,Low,Mid Floor
3,flat,sector 37c,3,3,3,Under Construction,1647.0,0,0,semifurnished,Low,Mid Floor
4,flat,sector 81,3,3,3+,Relatively New,1570.0,1,0,unfurnished,Low,Mid Floor
...,...,...,...,...,...,...,...,...,...,...,...,...
3479,flat,sector 92,3,3,3,Relatively New,1700.0,1,0,semifurnished,Low,Mid Floor
3480,flat,dwarka expressway,3,3,2,Under Construction,1303.0,0,0,semifurnished,Low,Mid Floor
3481,flat,sector 102,2,2,2,Relatively New,1230.0,0,0,semifurnished,Medium,Mid Floor
3482,flat,sector 111,3,4,2,Relatively New,2086.0,1,0,furnished,High,Mid Floor


In [229]:
with open('C:/Users/8429s/OneDrive/Desktop/Placement/Projects/Real Estate Price Prediction and Recommendation System/model/df.pkl', 'wb') as file:
    pickle.dump(X, file)

### Prediction

In [230]:
X.columns

Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category'],
      dtype='object')

In [231]:
X.iloc[0].values

array(['house', 'sector 4', np.int64(2), np.int64(2), '1',
       'Moderately Old', np.float64(67.0), np.int64(0), np.int64(0),
       'semifurnished', 'Low', 'Low Floor'], dtype=object)

In [232]:
data = [['house', 'sector 102', 4, 3, '3+', 'New Property', 2750, 0, 0, 'unfurnished', 'Low', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category']

# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 102,4,3,3+,New Property,2750,0,0,unfurnished,Low,Low Floor


In [233]:
np.expm1(final_pipe.predict(one_df))

array([2.8442411], dtype=float32)